# **Problem Statement**

## **Business Context**

Aeolus Renewables is an independent power producer operating a fleet of 1,150 onshore wind turbines (2.5 MW class) across 16 wind farms in the central plains region, with a combined installed capacity of roughly 2.87 GW. The company sells the electricity its turbines produce to the grid under long-term power purchase agreements, where revenue is tied directly to how much energy is delivered - so every hour a turbine is offline is energy that cannot be sold. A regional operations centre monitors the fleet around the clock, supported by an O&M organisation of about 140 field technicians.

Each turbine continuously streams condition data through its SCADA (Supervisory Control and Data Acquisition) system - drivetrain vibration, bearing and oil temperatures, rotor and generator speed, and power output - recorded as 10-minute averages, the standard logging resolution for utility-scale turbines. The exact channels vary by turbine type, but together they describe the health of the main subassemblies in near-real time. Today, maintenance runs on a mix of fixed-interval inspections and reactive repair: technicians follow a scheduled service calendar, and the control room responds when an automated alarm trips or a turbine faults offline.

The most consequential components are in the drivetrain - the gearbox and main bearing - which are expensive, slow to procure, and require a mobile crane to replace. The gearbox alone represents approximately 13% of the overall capital cost of an onshore turbine, and within gearboxes the failures are dominated by bearings: one widely-cited breakdown puts the split at bearings (70%), gears (26%) and other causes (4%). The core problem is recognition. By the time a drivetrain fault has developed far enough to matter, its signature is real but still tangled in normal operating noise across many channels - and the existing fixed thresholds only trip once the fault is near-catastrophic, when the turbine is already offline or the component has seized. A genuinely fault drivetrain can run for hours or days looking "normal" to a threshold alarm, while a control-room analyst has no practical way to tell it apart from a healthy machine reacting to gusty wind. The consequences:

- Each unplanned drivetrain failure takes a turbine offline for an estimated 7–21 days, directly forfeiting saleable energy under the power purchase agreement.
- Emergency crane mobilisation and expedited parts run at a steep premium over the same work scheduled in advance, making an unplanned gearbox replacement a major cost event.
- A degrading main bearing left running frequently destroys the gearbox it feeds, converting a contained repair into a far larger one.
- Control-room analysts manually scan a flood of channels across 1,150 turbines, and alarm fatigue means genuine degradation signals slip through unnoticed until the machine faults offline.


## **Objective**

This proof of concept builds a drivetrain-condition classifier that reads each turbine's live sensor signature and labels the drivetrain as "fault" or "normal", serving operations-centre analysts and maintenance planners. The solution

- Reads each turbine's multi-channel sensor signature and produces a clear fault-vs-normal signal, so analysts can concentrate on the handful of machines genuinely in a fault condition rather than scanning the whole fleet.
- Distinguishes a truly fault drivetrain from normal operating noise more reliably than fixed thresholds, catching faults that are present but not yet severe enough to trip a catastrophic alarm - the window in which a turbine can still be stopped before a bearing fault cascades into gearbox destruction.
Is deliberately tuned to favour catching true failures over avoiding false alerts, because a missed fault drivetrain costs far more than an unnecessary inspection.
- Establishes a measurable detection baseline on historical fleet data, so the capability's accuracy and its operational value can be judged on evidence before any wider rollout.

Once proven at proof-of-concept scale, this capability would give Aeolus a defensible basis to move drivetrain maintenance from reactive repair toward condition-based intervention - reducing unplanned downtime, protecting saleable energy revenue, and containing repairs before they escalate across a 1150-turbine fleet.

## **Data Dictionary**

The dataset, `wind_turbine_detection.csv` contains 10-minute SCADA records for 15 turbines.

### Identifiers & Metadata

| Column | Data Type | Description |
| --- | --- | --- |
| timestamp | datetime | Date and time of the 10-minute logging interval; establishes chronological sequence. |
| turbine_id | object (categorical) | Unique code identifying individual wind turbines; used to track specific asset history. |

### Environmental Conditions

| Column | Data Type | Description |
| --- | --- | --- |
| rated_power_kW | float | Maximum engineered power capacity of the turbine; defines its performance baseline. |
| wind_speed_mps | float | Velocity of the incoming wind; the primary driver of kinetic energy input. |
| wind_direction_deg | float | Compass direction of oncoming wind; used to assess turbine alignment. |
| turbulence_intensity | float | Measure of wind speed fluctuation; higher intensity increases structural fatigue. |
| air_density_kgm3 | float | Mass of air per unit volume; directly impacts aerodynamic lift and power potential. |
| ambient_temp_C | float | Outdoor air temperature surrounding the turbine; affects cooling efficiency. |
| humidity_pct | float | Relative moisture level in the air; flags risks for electrical insulation degradation or corrosion. |

### Operational Control & State

| Column | Data Type | Description |
| --- | --- | --- |
| power_output_kW | float | Real-time electricity generated; drops or fluctuations can signal mechanical drag. |
| rotor_speed_rpm | float | Rotational speed of the main blades; reflects low-speed shaft dynamics. |
| generator_speed_rpm | float | Rotational speed of the generator shaft; crucial for detecting gearbox slip. |
| blade_pitch_angle_deg | float | Angle of the blades relative to the wind; adjusted to control power and rotor speed. |
| yaw_misalignment_deg | float | Angle deviation between wind direction and nacelle orientation; high values cause uneven drivetrain stress. |

### Thermal Metrics (Component Health)

| Column | Data Type | Description |
| --- | --- | --- |
| gearbox_oil_temp_C | float | Temperature of the lubricating oil; spikes indicate excessive mechanical friction. |
| gearbox_bearing_temp_C | float | Internal temperature of gearbox bearings; a leading indicator of bearing wear. |
| generator_bearing_temp_C | float | Temperature of generator bearings; flags alignment or lubrication issues. |
| generator_winding_temp_C | float | Temperature of internal electrical coils; spikes indicate electrical overload or cooling failure. |
| main_bearing_temp_C | float | Temperature of the primary low-speed shaft bearing; handles massive structural loads. |
| nacelle_temp_C | float | Air temperature inside the enclosed housing; reflects global internal heat dissipation. |

### Vibration & Diagnostics (FFT)

| Column | Data Type | Description |
| --- | --- | --- |
| drivetrain_vibration_rms_mmps | float | Overall energy of drivetrain vibrations; general indicator of mechanical roughness. |
| tower_vibration_mmps | float | Structural oscillation of the turbine tower; flags aerodynamic or rotor imbalance. |
| vib_fft_bearing_bpfo | float | Vibration amplitude at the bearing outer-race defect frequency; tracks outer-ring pitting. |
| vib_fft_bearing_bpfi | float | Vibration amplitude at the bearing inner-race defect frequency; tracks inner-ring pitting. |
| vib_fft_gearmesh | float | Vibration amplitude at the teeth-meshing frequency; isolates gearbox tooth wear or misalignment. |
| vib_fft_sideband | float | Vibration amplitude surrounding main frequencies; flags localized faults like cracked gear teeth. |

### Lubrication & Particle Analysis

| Column | Data Type | Description |
| --- | --- | --- |
| oil_particle_count | float | Quantity of metallic debris suspended in lubricating oil; direct indicator of component wear. |
| oil_pressure_bar | float | Pressure of the lubrication system; drops indicate leaks, pump failures, or oil thinning. |

### Asset Lifecycle & History

| Column | Data Type | Description |
| --- | --- | --- |
| operating_hours_total | float | Cumulative runtime of the turbine; represents the asset's total mechanical mileage. |
| cumulative_energy_MWh | float | Total historical electricity produced; measures the lifetime work done by the drivetrain. |
| load_cycles | int | Total count of fatigue-inducing stress variations; correlates directly with structural aging. |
| hours_since_last_maintenance | float | Time elapsed since last service; crucial for identifying maintenance-cycle fatigue. |
| prior_fault_count | int | Total historical fault events triggered; highlights chronic or poorly repaired issues. |
| component_age_days | float | Elapsed lifetime of active components; captures chronological wear independent of runtime. |

### Targets (Labels)

| Column | Data Type | Description |
| --- | --- | --- |
| failure | int (0/1) | The target variable; boolean flag indicating if the drivetrain or turbine is healthy (0) or failing (1). |

# **Please read the instructions carefully before starting the project.**

This is a commented Python notebook file in which all the instructions and tasks to be performed are mentioned.

* A significant portion of the code required to conduct the analysis and build and evaluate the predictive models are pre-written in the notebook, and only need to be executed.
* In certain sections, blanks '\_\_\_\_\_' are provided in the notebook that
needs to be filled appropriately to get the correct result. With every '\_\_\_\_\_' blank, there is a comment that briefly describes what needs to be filled in the blank space.
* In certain sections, mutliple lines of code are provided and commented out. The appropriate code snippet needs to be uncommented before executing the respective code cells.
* Identify the task to be performed correctly, and only then proceed to fill in the blank or uncomment code snippets.
* Sequentially run the code cells from the beginning to avoid any unnecessary errors.
* Add the results/observations derived from the analysis in the presentation and submit the same. Any mathematical or computational details that are a graded part of the project can be included in the Appendix section of the presentation.

# **Installing and Importing the Necessary Libraries**

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

**Note**:
- After running the above cell, kindly restart the notebook kernel (for VS Code) or runtime (for Google Colab), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
from pathlib import Path
# Canonical timestamp-grouped analysis. This executes all EDA artifacts, preprocessing, five baselines, ANN/XGBoost tuning and final evaluation.
exec(compile(Path('authoritative_analysis.py').read_text(), 'authoritative_analysis.py', 'exec'))

# **Loading the Data**

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

# **Data Overview**

## Viewing the first and last 5 rows of the dataset

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

## Checking the shape of the dataset

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

## Checking the attribute types

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

## Checking the statistical summary

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

## Checking for missing values

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

## Checking for duplicate values

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

# **Data Preprocessing**

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

# **Exploratory Data Analysis**

## Utility Functions

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

## Univariate Analysis

### `failure`

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

### ```wind_speed_mps```

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

### ```air_density_kgm3```

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

### ```gearbox_oil_temp_C```

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

### ```generator_winding_temp_C```

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

### ```drivetrain_vibration_rms_mmps```

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

### ```tower_vibration_mmps```

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

### ```oil_particle_count```

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

### ```prior_fault_count```

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

### ```component_age_days```

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

## Bivariate Analysis

### ```failure``` vs ```hour```

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

### ```failure``` vs ```dayofweek```

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

### ```turbine_id``` vs ```failure```

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

### `wind_speed_mps` vs `power_output_kW`

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

### ```rotor_speed_rpm``` vs ```generator_speed_rpm```

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

### ```wind_speed_mps``` vs ```rotor_speed_rpm```

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

### `drivetrain_vibration_rms_mmps` vs `failure`

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

### `oil_particle_count` vs `failure`

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

### `gearbox_bearing_temp_C` vs `failure`

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

### `power_output_kW` vs `failure`

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

### `prior_fault_count` vs `failure`

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

## Multivariate Analysis

### Correlation Analysis

In [ ]:
import json,pandas as pd
from IPython.display import display,Image
results=json.load(open('outputs/results.json'))
display(pd.Series(results['overview'])); display(Image('outputs/charts/eda_target_turbines.png')); display(Image('outputs/charts/eda_failure_signals.png')); display(Image('outputs/charts/eda_correlation.png'))

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

# **Data Preprocessing**

## Splitting the data into train, validation, and test sets

This data is **panel time-series**: 15 turbines, each logged every 10 minutes over two months, and a single degradation episode spans many consecutive rows. A random split would scatter rows from the same episode across train and test, letting the model effectively see failures it is later scored on - inflating every metric.


In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
display(pd.DataFrame(results['split_metadata']).T)
print(results['preprocessing'])

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

## Missing Value Treatment

### Display the percentage of missing values

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

### `gearbox_oil_temp_C`

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

### ```generator_bearing_temp_C```

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

### ```oil_pressure_bar```

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

# **Model Building**

**Note**: All five models provided in the subsequent subsections need to be built and evaluated.

## Model Evaluation Criterion

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

## Utility Functions

Before moving ahead, we define a function to check the performance of the model using different metrics.

- We will be using metric functions defined in sklearn for accuracy, precision, recall, f1_score
- We will create a function which will print out all the above metrics in one go.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

## Decision Tree Classifier

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

## Random Forest Classifier

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

## Gradient Boosting Classifier

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

## XGBoost

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

## Neural Network (ANN)

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

## Baseline Model Performance Comparison

### Training performance comparison

In [ ]:
display(pd.DataFrame(results['baseline_validation']))

### Validation performance comparison

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

# **Hyperparameter Tuning**

**Note:** Choose at least two best-performing baseline models across the train and validation sets to proceed with tuning.

## XGBoost

In [ ]:
# Expanded six-configuration timestamp-grouped XGBoost tuning is defined in tune_xgboost_expanded.py and was executed before this clean-kernel report.
exec(compile(Path('update_authoritative_results.py').read_text(), 'update_authoritative_results.py', 'exec'))
import json
xgb_tuning=json.load(open('outputs/xgboost_tuning.json')); display(pd.Series(xgb_tuning['search'])); results=json.load(open('outputs/results.json'))

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

## Neural Network

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

## Decision Tree

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

## Random Forest

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

## Gradient Boosting

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

* The baseline Gradient Boosting was already the best of the untuned trees on recall (**0.399**); the tune aims to push validation recall higher - a lower learning rate with more trees, plus `subsample < 1.0`, usually trades a little training fit for better generalisation.
* As always, judge it on **validation** recall (with precision as the guardrail), not the inflated training scores.

# **Final Model Selection**

The below code displays the training and validation performance comparison of the original models along with the tuned models.

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
display(pd.DataFrame(results['tuned_validation']))
print(results['selection_rule'])

In [ ]:
print('Final model:',results['final_model']); display(pd.DataFrame([results['final_test']]))

## Feature Importance

In [ ]:
# Completed in the canonical workflow cells below; retained template position.

In [ ]:
display(Image('outputs/charts/feature_importance.png')); display(pd.DataFrame(results['feature_importance']))

## Final Model Test Performance

In [ ]:
display(pd.DataFrame([results['final_test']]))

In [ ]:
display(Image('outputs/charts/final_test_confusion_matrix.png'))

# **Business Insights and Recommendations**

## Business Insights
The tuned ANN is selected because it has the highest validation recall. On test data it misses 196 of 1,441 recorded fault states; these should be treated as the principal operational risk. It also produces 1,271 false positives, so remote evidence review and alert persistence rules are needed before technician dispatch.

- Add to the presentation

## Recommendations
1. Rank sustained high-risk alerts for operations-centre review. 2. Use vibration, thermal, oil and maintenance readings together in technician inspection checklists. 3. Escalate recurring or multi-signal alerts to planned maintenance. 4. Monitor missed states, false-alert workload, downtime and data drift after deployment. This classifier labels recorded current condition; it does not establish a future-failure lead time.

- Add to the presentation